## Purpose

This notebook has one goal: **discover the real column names** in each source file.

All `src/transform/` modules use a `column_map` parameter to rename raw source columns
to canonical silver-layer names. The default maps live in `settings.py` as placeholders.
Before EDA confirms the actual names, transform modules log warnings and may skip joins.

**Workflow:**
1. Run Step 0 to download raw files (one-time, ~145 MB total)
2. Run each exploration section — inspect column names printed to output
3. Fill in the discovered names in the `*_COLUMN_MAP_DISCOVERED` dicts at the bottom of each section
4. Copy confirmed maps into `src/config/settings.py` (the four `*_COLUMN_MAP` dicts)

**Why this matters:** The French Interior Ministry does not publish a public field dictionary
for its election CSV files. Column names change between election cycles.
Discovering them empirically before writing transform logic prevents hard-to-debug
"column not found" errors downstream.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Add project root to path so we can import from src/
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config.settings import DATA_SOURCES, RAW_DIR

print('Project root:', project_root)
print('RAW_DIR:', RAW_DIR)

## Step 0: Download raw files (run once)

Run the cells below to download all source files to `data/raw/`.
Each function is idempotent — re-running skips the download if the MD5 hash is unchanged.

In [ ]:
# Uncomment to download all 5 source files
# from src.ingest.candidates import ingest_candidates, ingest_candidates_tour2
# from src.ingest.geography import ingest_geography
# from src.ingest.incumbents import ingest_incumbents
# from src.ingest.seats import ingest_seats_population

# ingest_candidates()        # 137.8 MB — Tour 1 (analysis subject pool)
# ingest_candidates_tour2()  # 20.5 MB  — Tour 2 (advanced_to_tour2 flag only)
# ingest_geography()         # 3 MB
# ingest_incumbents()        # 3.9 MB
# ingest_seats_population()  # 2.5 MB XLSX

## 1. Tour 1 Candidates — 137.8 MB CSV

**Schema confirmed by EDA (2026-03-21):** 17 columns, semicolon separator, latin-1 encoding.

| Question | Answer |
|---|---|
| List-leader flag | `Tête de liste` — values `'Oui'`/`'Non'` |
| Gender column | `Sexe` — values `'M'`/`'F'` |
| Commune code | `Code circonscription` — 5-digit INSEE string |
| Nuance code | `Code nuance de liste` — maps to `NUANCE_GROUP_MAP` |
| Position on list | `Ordre` — integer; 1 = leader (secondary to `Tête de liste` flag) |

**Note:** Tour 1 and Tour 2 share an identical schema — verified in Section 2 below.

In [ ]:
candidates_path = RAW_DIR / 'gouv' / 'candidates_tour1.csv'
print(f'File exists: {candidates_path.exists()}')
print(f'File size: {candidates_path.stat().st_size / 1e6:.1f} MB' if candidates_path.exists() else 'NOT FOUND')

In [ ]:
# Read only 5 rows to see column names without loading 137 MB
try:
    candidates_sample_df = pd.read_csv(
        candidates_path,
        nrows=5,
        dtype=str,
        encoding='utf-8',
        sep=None,
        engine='python',
    )
    print('\n=== CANDIDATES COLUMNS ===')
    for i, col in enumerate(candidates_sample_df.columns):
        print(f'  [{i:2d}] {col!r}')
    print(f'\nTotal columns: {len(candidates_sample_df.columns)}')
    print('\nFirst 5 rows:')
    display(candidates_sample_df)
except Exception as e:
    print(f'ERROR: {e}')

In [ ]:
# Confirmed column names from EDA output above.
# Copied into src/config/settings.py CANDIDATES_COLUMN_MAP on 2026-03-21.

CANDIDATES_COLUMN_MAP_DISCOVERED = {
    "Code département":                "dep_code",
    "Département":                     "dep_name",
    "Code circonscription":            "commune_insee",       # 5-digit INSEE — confirmed
    "Circonscription":                 "commune_name",
    "Numéro de panneau":               "list_id",
    "Libellé abrégé de liste":         "list_label_short",
    "Libellé de la liste":             "list_label",
    "Code nuance de liste":            "list_nuance",          # key into NUANCE_GROUP_MAP
    "Nuance de liste":                 "list_nuance_label",
    "Tête de liste":                   "is_list_leader",       # 'Oui'/'Non' — authoritative flag
    "Ordre":                           "position_on_list",     # integer rank; 1 = leader
    "Sexe":                            "gender",               # 'M' or 'F'
    "Nom sur le bulletin de vote":     "family_name",
    "Prénom sur le bulletin de vote":  "given_name",
    "Nationalité":                     "nationality",
    "Code personnalité":               "personality_code",
    "CC":                              "candidate_category",   # semantics unknown; preserved
}
print("CANDIDATES_COLUMN_MAP confirmed ✓  (17 columns, same schema for Tour 1 and Tour 2)")

### Gender values — confirmed

Column `Sexe` contains values `'M'` and `'F'` — confirmed by EDA.
No recode step is needed in `src/transform/dim_candidate.py`.

The cell below shows the actual value distribution as a reference.

In [ ]:
# Gender column confirmed as 'Sexe' — directly verify value distribution.
try:
    gender_check_df = pd.read_csv(
        candidates_path,
        nrows=1000,
        dtype=str,
        encoding='latin-1',   # confirmed encoding for Interior Ministry files
        sep=None,
        engine='python',
    )
    if 'Sexe' in gender_check_df.columns:
        print("Gender column: 'Sexe'")
        print(gender_check_df['Sexe'].value_counts())
        print("\nExpected: 'M' and 'F' only. If other values appear, add a recode step in dim_candidate.py.")
    else:
        print("WARNING: 'Sexe' not found. Available columns:", list(gender_check_df.columns))
except Exception as e:
    print(f'ERROR: {e}')


## 2. Tour 2 Candidates — 20.5 MB CSV

Tour 2 data is used **only** to derive the `advanced_to_tour2` control flag in `dim_candidate_leader`.
It is **not** a second set of analysis subjects — see `src/ingest/candidates.py` module docstring
for the survivorship-bias rationale.

**Questions to answer:**
- Does Tour 2 have the same schema as Tour 1? (expected: yes — same Interior Ministry format)
- What column identifies tête de liste (position == 1 on the list)?
- Are there any extra columns in Tour 2 not present in Tour 1?

In [ ]:
tour2_path = RAW_DIR / 'gouv' / 'candidates_tour2.csv'
print(f'File exists: {tour2_path.exists()}')
print(f'File size: {tour2_path.stat().st_size / 1e6:.1f} MB' if tour2_path.exists() else 'NOT FOUND')

try:
    tour2_sample_df = pd.read_csv(
        tour2_path,
        nrows=5,
        dtype=str,
        encoding='utf-8',
        sep=None,
        engine='python',
    )
    print('\n=== TOUR 2 CANDIDATES COLUMNS ===')
    for i, col in enumerate(tour2_sample_df.columns):
        print(f'  [{i:2d}] {col!r}')

    # Quick check: does Tour 2 have the same schema as Tour 1?
    tour1_sample_df = pd.read_csv(
        RAW_DIR / 'gouv' / 'candidates_tour1.csv',
        nrows=1,
        dtype=str,
        encoding='utf-8',
        sep=None,
        engine='python',
    )
    t1_cols = set(tour1_sample_df.columns)
    t2_cols = set(tour2_sample_df.columns)
    print(f'\nColumns in Tour 2 but NOT in Tour 1: {t2_cols - t1_cols}')
    print(f'Columns in Tour 1 but NOT in Tour 2: {t1_cols - t2_cols}')
    print('\nFirst 5 rows of Tour 2:')
    display(tour2_sample_df)
except Exception as e:
    print(f'ERROR: {e}')

## 3. Seats / Population — 2.5 MB XLSX

This is the only non-CSV source file. Key columns needed:
- `commune_insee` — join key (5-character code, must stay as string to preserve leading zeros)
- `seats_municipal` — seats to fill in the municipal council (`NBRE_SAP_COM` or similar)
- `seats_epci` — community council seats (`NBRE_SAP_EPCI` or similar)
- `population` — INSEE resident population used for city-size stratification

**Important:** French government XLSX files often have a legend/header sheet before the data.
Check whether data is on sheet index 0, 1, or later — record the correct `SEATS_SHEET_NAME` below.

In [ ]:
seats_path = RAW_DIR / 'gouv' / 'seats_population.xlsx'
print(f'File exists: {seats_path.exists()}')
if seats_path.exists():
    print(f'File size: {seats_path.stat().st_size / 1e6:.1f} MB')

try:
    # Probe up to 3 sheet indices — data may not be on sheet 0
    for sheet_idx in range(3):
        try:
            seats_sample_df = pd.read_excel(
                seats_path,
                sheet_name=sheet_idx,
                nrows=5,
                dtype=str,
                engine='openpyxl',
            )
            print(f'\n=== SHEET {sheet_idx} COLUMNS ===')
            for i, col in enumerate(seats_sample_df.columns):
                print(f'  [{i:2d}] {col!r}')
            print(f'\nFirst 5 rows (sheet {sheet_idx}):')
            display(seats_sample_df)
        except Exception as sheet_err:
            print(f'Sheet {sheet_idx}: ERROR — {sheet_err}')
            break
except Exception as e:
    print(f'ERROR: {e}')

In [ ]:
# Confirmed column names from EDA output above.
# Copied into src/config/settings.py SEATS_COLUMN_MAP on 2026-03-21.

SEATS_COLUMN_MAP_DISCOVERED = {
    "CODE_DPT":          "dep_code",
    "LIB_DPT":           "dep_name",
    "CODE_EPCI":         "epci_code",
    "LIB_EPCI":          "epci_name",
    "CODE_COMMUNE":      "commune_insee",           # join key to COG; must stay as string
    "LIB_COMMUNE":       "commune_name",
    "CODE_COM_ASSOCIEE": "associated_commune_code",
    "LIB_COM_ASSOCIEE":  "associated_commune_name",
    "NBRE_BV":           "polling_station_count",
    "POPULATION":        "population",              # resident pop — city-size stratification
    "INSCRITS":          "registered_voters",
    "NBRE_SAP_COM":      "seats_municipal",
    "NBRE_SAP_EPCI":     "seats_epci",
}
SEATS_SHEET_NAME = 0  # Data is on sheet 0 — confirmed by EDA
print("SEATS_COLUMN_MAP confirmed ✓  (13 columns, sheet_name=0)")

## 4. RNE Incumbent Mayors — 3.9 MB CSV

RNE = Répertoire National des Élus (National Elected Officials Registry).
This snapshot lists outgoing mayors from the 2020–2026 term.

Used to derive `is_incumbent` in `dim_candidate_leader` via fuzzy name matching
(`rapidfuzz.fuzz.token_sort_ratio` — word-order invariant). Key columns needed:
- Family name and given name (or a combined full name field)
- Commune INSEE code — join key for pre-filtering before fuzzy matching
- Mandate role — to confirm the person is a mayor, not just a councillor

**Why fuzzy matching?** The Interior Ministry candidate file and the RNE use different
name formats (family-given vs. given-family) and may have accents handled differently.
`token_sort_ratio` is invariant to word order and accent normalization handles encoding.

In [ ]:
rne_path = RAW_DIR / 'rne' / 'rne_incumbents.csv'
print(f'File exists: {rne_path.exists()}')
if rne_path.exists():
    print(f'File size: {rne_path.stat().st_size / 1e6:.1f} MB')

try:
    rne_sample_df = pd.read_csv(
        rne_path,
        nrows=10,
        dtype=str,
        encoding='utf-8',
        sep=None,
        engine='python',
    )
    print('\n=== RNE COLUMNS ===')
    for i, col in enumerate(rne_sample_df.columns):
        print(f'  [{i:2d}] {col!r}')
    print(f'\nTotal columns: {len(rne_sample_df.columns)}')
    print('\nFirst 10 rows:')
    display(rne_sample_df)
    # NOTE: Source is mun2026-maires-sortants-20260227.csv (mayors-only extract).
    # There is NO function/role column because all rows are mayors by definition.
    # No mandate-role filter is needed in dim_candidate.py.
except Exception as e:
    print(f'ERROR: {e}')

# ── Row-count sanity check ───────────────────────────────────────────────
# Confirms we downloaded mun2026-maires-sortants (mayors only), not cm-sortants.
# maires file: ~34,900 rows (one per French commune)
# cm file:     ~400,000+ rows (all municipal councillors)
rne_full_df = pd.read_csv(rne_path, dtype=str, encoding='utf-8', sep=None, engine='python')
row_count = len(rne_full_df)
print(f'\nTotal RNE rows: {row_count:,}')
if row_count < 50_000:
    print('OK: mayors-only file confirmed — no function filter needed in dim_candidate.py')
else:
    print('WARNING: row count is high — may have downloaded the wrong file (cm-sortants).')
    print('  Check DATA_SOURCES["rne_incumbents"] URL in settings.py.')


In [ ]:
# Confirmed column names from EDA output above.
# Source file: mun2026-maires-sortants-20260227.csv (mayors only — no function column needed).
# Copied into src/config/settings.py RNE_COLUMN_MAP on 2026-03-21.

RNE_COLUMN_MAP_DISCOVERED = {
    "Code du département":                              "dep_code",
    "Libellé du département":                           "dep_name",
    "Code de la collectivité à statut particulier":     "special_collectivity_code",
    "Libellé de la collectivité à statut particulier":  "special_collectivity_name",
    "Code de la commune":                               "commune_insee",   # join key for fuzzy matching
    "Libellé de la commune":                            "commune_name",
    "Nom de l'élu":                                     "family_name",
    "Prénom de l'élu":                                  "given_name",
    "Code sexe":                                        "gender",
    "Date de naissance":                                "birth_date",
    "Code de la catégorie socio-professionnelle":       "socio_professional_code",
    "Libellé de la catégorie socio-professionnelle":    "socio_professional_label",
    "Date de début du mandat":                          "mandate_start_date",
    "Date de début de la fonction":                     "function_start_date",
}
print("RNE_COLUMN_MAP confirmed ✓  (14 columns, mayors-only file — no function filter needed)")

## 5. INSEE COG 2026 Communes — 3 MB CSV

COG = Code Officiel Géographique — the canonical geographic reference for France.
Unlike the Interior Ministry files, COG column names are well-documented, but we verify them anyway.

Key columns needed:
- `COM` — commune INSEE code (5 characters, must keep as string to preserve leading zeros)
- `LIBELLE` — commune name
- `DEP` — department code
- `REG` — region code (used for geographic diversity check in sampling)
- `TYPECOM` — locality type: filter to `'COM'` (actual communes only)

TYPECOM values to expect:
- `COM` — standard commune (what we want)
- `ARM` — arrondissement municipal (Paris/Lyon/Marseille sub-units, excluded)
- `COMA` — commune associée (historical, excluded)
- `COMD` — commune déléguée (merged commune sub-unit, excluded)

In [ ]:
cog_path = RAW_DIR / 'insee' / 'insee_cog_communes.csv'
print(f'File exists: {cog_path.exists()}')
if cog_path.exists():
    print(f'File size: {cog_path.stat().st_size / 1e6:.1f} MB')

try:
    cog_sample_df = pd.read_csv(
        cog_path, nrows=10, dtype=str, encoding='utf-8', sep=None, engine='python',
    )
    print('\n=== COG COLUMNS ===')
    for i, col in enumerate(cog_sample_df.columns):
        print(f'  [{i:2d}] {col!r}')
    print('\nFirst 10 rows (verify COM, DEP, REG, LIBELLE, TYPECOM):')
    display(cog_sample_df)

    # Load full file — cog_full_df is referenced by the JOIN validation cell at the end
    cog_full_df = pd.read_csv(
        cog_path, dtype=str, encoding='utf-8', sep=None, engine='python',
    )
    typecom_col = next((c for c in cog_full_df.columns if 'TYPECOM' in c.upper()), None)
    if typecom_col:
        print(f'\nTYPECOM distribution (column: {typecom_col!r}):')
        print(cog_full_df[typecom_col].value_counts())
        com_count = (cog_full_df[typecom_col] == 'COM').sum()
        print(f'\nRows with TYPECOM == "COM" (actual communes kept in silver): {com_count:,}')
except Exception as e:
    print(f'ERROR: {e}')


In [ ]:
# Confirmed column names from EDA output above.
# COG column names are stable across years — pre-filled from INSEE docs and verified.
# Copied into src/config/settings.py COG_COLUMN_MAP on 2026-03-21.

COG_COLUMN_MAP_DISCOVERED = {
    "COM":     "commune_insee",   # 5-char code; filter TYPECOM == 'COM' in silver transform
    "LIBELLE": "commune_name",
    "DEP":     "dep_code",
    "REG":     "reg_code",
    "TYPECOM": "typecom",         # 'COM'/'ARM'/'COMA'/'COMD' — filter to 'COM' only
}
print("COG_COLUMN_MAP confirmed ✓  (5 columns used; file has 12 total)")

## Summary: settings.py updates needed

After filling in all five `*_COLUMN_MAP_DISCOVERED` dicts above, run the print cell below
to get a copy-paste-ready output, then update the four maps in `src/config/settings.py`.

In [ ]:
print('=== COPY INTO settings.py ===')
print()
print('COG_COLUMN_MAP:', COG_COLUMN_MAP_DISCOVERED)
print('SEATS_COLUMN_MAP:', SEATS_COLUMN_MAP_DISCOVERED)
print('CANDIDATES_COLUMN_MAP:', CANDIDATES_COLUMN_MAP_DISCOVERED)
print('RNE_COLUMN_MAP:', RNE_COLUMN_MAP_DISCOVERED)
print()
print(f'seats sheet_name to use: {SEATS_SHEET_NAME}')

## Commune JOIN key validation

This section is at the **end of the notebook** because it needs two DataFrames
that are defined in earlier sections:
- `seats_sample_df` — loaded in Section 3 (Seats exploration cell)
- `cog_full_df` — loaded in Section 5 (COG exploration cell, just above)

Run Sections 3 and 5 first, then run the cell below.
It verifies that `CODE_COMMUNE` (seats) and `COM` (COG) use the same 5-character
zero-padded format — if they don’t match, `dim_commune` will have NULL population values.

In [ ]:
# Column names are confirmed. Run this to verify the COG <-> seats join key.

COG_COMMUNE_COL   = 'COM'           # confirmed by EDA
SEATS_COMMUNE_COL = 'CODE_COMMUNE'  # confirmed by EDA (NOT 'CODECOMMUNE')

if 'cog_full_df' not in dir() or 'seats_sample_df' not in dir():
    print('Run Section 3 (Seats) and Section 5 (COG) exploration cells first.')
else:
    cog_codes   = set(cog_full_df[COG_COMMUNE_COL].dropna().unique())
    seats_codes = set(seats_sample_df[SEATS_COMMUNE_COL].dropna().unique())

    in_both       = len(cog_codes & seats_codes)
    in_cog_only   = len(cog_codes - seats_codes)
    in_seats_only = len(seats_codes - cog_codes)

    print(f'Communes in both:                          {in_both:,}')
    print(f'In COG only (NULL after left join):        {in_cog_only:,}')
    print(f'In seats only (no COG entry):              {in_seats_only:,}')
    print('\nExpected: in_both ~34,900; in_seats_only should be 0 or near-0.')
